<a href="https://colab.research.google.com/github/Saadhvi-29/GEN_AI_LAB/blob/main/GenAI_Lab11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install langchain faiss-cpu sentence-transformers pillow transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 39.3 MB/s eta 0:00:00


In [6]:
# -----------------------------
# Install Required Libraries
# -----------------------------
import subprocess, sys

packages = [
    "sentence-transformers",
    "faiss-cpu",
    "pillow",
    "numpy",
    "transformers"
]

for p in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", p])


# -----------------------------
# Imports
# -----------------------------
import urllib.request
from PIL import Image
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import pipeline


# -----------------------------
# Step 1: Download Images
# -----------------------------
def download_image(url, filename):
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0"}
    )
    with urllib.request.urlopen(req) as response:
        with open(filename, "wb") as f:
            f.write(response.read())


image_urls = {
    "diagram.png": "https://images.unsplash.com/photo-1555949963-aa79dcee981c",
    "login_ui.png": "https://images.unsplash.com/photo-1563986768609-322da13575f3"
}

for filename, url in image_urls.items():
    download_image(url, filename)

print("Images downloaded successfully")


# -----------------------------
# Step 2: Text Documents
# -----------------------------
texts = [
    "This diagram explains the architecture of a neural network.",
    "This screenshot shows a login interface used in applications.",
    "Tables in reports show structured financial data."
]


# -----------------------------
# Step 3: Load Multimodal Model
# -----------------------------
model = SentenceTransformer("clip-ViT-B-32")


# -----------------------------
# Step 4: Create Embeddings
# -----------------------------
text_embeddings = model.encode(texts)

image_paths = ["diagram.png", "login_ui.png"]
image_embeddings = []

for path in image_paths:
    img = Image.open(path).convert("RGB")
    emb = model.encode(img)
    image_embeddings.append(emb)

image_embeddings = np.array(image_embeddings)


# -----------------------------
# Step 5: Combine Embeddings
# -----------------------------
all_embeddings = np.vstack([text_embeddings, image_embeddings])


# -----------------------------
# Step 6: Create Vector DB
# -----------------------------
dimension = all_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(all_embeddings)

print("Vector database created")


# -----------------------------
# Step 7: Query
# -----------------------------
query = "Explain neural network architecture diagram"

query_embedding = model.encode([query])

k = 3
distances, indices = index.search(query_embedding, k)


# -----------------------------
# Step 8: Retrieve Context
# -----------------------------
context = ""

print("\nRetrieved Information:\n")

for i in indices[0]:

    if i < len(texts):
        print("Text:", texts[i])
        context += texts[i] + "\n"

    else:
        img_index = i - len(texts)
        print("Image:", image_paths[img_index])
        context += f"Image file: {image_paths[img_index]}\n"


# -----------------------------
# Step 9: Generation (LLM)
# -----------------------------
generator = pipeline("text-generation", model="gpt2")

prompt = f"""
Use the following retrieved information to answer the question.

Context:
{context}

Question: {query}

Answer:
"""

response = generator(prompt, max_length=200)

print("\nGenerated Answer:\n")
print(response[0]["generated_text"])

Images downloaded successfully


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: /root/.cache/huggingface/hub/models--sentence-transformers--clip-ViT-B-32/snapshots/327ab6726d33c0e22f920c83f2ff9e4bd38ca37f/0_CLIPModel
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector database created

Retrieved Information:

Text: This diagram explains the architecture of a neural network.
Text: Tables in reports show structured financial data.
Text: This screenshot shows a login interface used in applications.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generated Answer:


Use the following retrieved information to answer the question.

Context:
This diagram explains the architecture of a neural network.
Tables in reports show structured financial data.
This screenshot shows a login interface used in applications.


Question: Explain neural network architecture diagram

Answer:

Cynopsis is a neural network architecture, which is a set of general-purpose neural networks with a set of fixed-state representations.

The primary function of neural networks is to provide basic information about the state of the situation.

More than one type of neural network can be introduced in a single application.

The main purpose of a neural network is to provide information about the state of the situation.

The state of the situation is represented by a set of neurons.

The neural network architecture consists of a set of neurons.

Each neuron corresponds to a set of connections of the neural network.

The connections of the neural network are rep

In [5]:
# -----------------------------
# Install Libraries
# -----------------------------
import subprocess, sys

packages = [
    "sentence-transformers",
    "faiss-cpu",
    "numpy",
    "transformers"
]

for p in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", p])


# -----------------------------
# Imports
# -----------------------------
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import pipeline


# -----------------------------
# Step 1: Create Knowledge Sources
# -----------------------------

internal_docs = [
    "Employees are entitled to 20 days of paid leave per year.",
    "Travel reimbursement must be submitted within 7 days.",
    "The company follows a hybrid work policy."
]

faq_docs = [
    "Shipping usually takes 3 to 5 business days.",
    "Refunds are available within 30 days of purchase.",
    "Customer support is available 24 hours via email."
]


# -----------------------------
# Step 2: Load Embedding Model
# -----------------------------
model = SentenceTransformer("all-MiniLM-L6-v2")


# -----------------------------
# Step 3: Create Embeddings
# -----------------------------
internal_embeddings = model.encode(internal_docs)
faq_embeddings = model.encode(faq_docs)


# -----------------------------
# Step 4: Create Vector Databases
# -----------------------------
dim = internal_embeddings.shape[1]

internal_index = faiss.IndexFlatL2(dim)
internal_index.add(np.array(internal_embeddings))

faq_index = faiss.IndexFlatL2(dim)
faq_index.add(np.array(faq_embeddings))

print("Vector databases created")


# -----------------------------
# Step 5: Agent Router
# -----------------------------
def router(query):

    query = query.lower()

    if "employee" in query or "policy" in query or "leave" in query:
        return "internal"

    elif "shipping" in query or "refund" in query or "support" in query:
        return "faq"

    else:
        return "faq"


# -----------------------------
# Step 6: User Query
# -----------------------------
query = "What is the employee leave policy?"

print("\nUser Query:", query)


# -----------------------------
# Step 7: Agent Chooses Source
# -----------------------------
source = router(query)

print("Agent selected source:", source)


# -----------------------------
# Step 8: Retrieval
# -----------------------------
query_embedding = model.encode([query])

if source == "internal":

    distances, indices = internal_index.search(query_embedding, 2)

    context = ""
    for i in indices[0]:
        context += internal_docs[i] + "\n"

else:

    distances, indices = faq_index.search(query_embedding, 2)

    context = ""
    for i in indices[0]:
        context += faq_docs[i] + "\n"


print("\nRetrieved Context:\n")
print(context)


# -----------------------------
# Step 9: Generation
# -----------------------------
generator = pipeline("text-generation", model="gpt2")

prompt = f"""
Use the following context to answer the question.

Context:
{context}

Question: {query}

Answer:
"""

response = generator(prompt, max_length=150)

print("\nGenerated Answer:\n")
print(response[0]["generated_text"])

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generated Answer:


Context:
This diagram explains the architecture of a neural network.
Tables in reports show structured financial data.
This screenshot shows a login interface used in applications.


Question: show me neural network architecture

Answer:

The diagram shows how a neural network is structured.

The diagram shows how it is structured.

It is not a linear pattern.

The only difference between the diagrams is the shape of some nodes.

The shape of the network is not affected by the type of data being collected.

Data are not recorded.

The data are not processed in any way.

The network is not affected by the presence of any external factors.

The network is not affected by any external factors.

In the diagram, the network is divided into 10 "states" (one for each individual node), each state having its own "connections" between the nodes.

Each state has its own "connections" with the other nodes, each with its own "connections" with the other nodes.

The graph shows 

In [7]:
# -----------------------------
# Install Libraries
# -----------------------------
import subprocess, sys

packages = [
    "sentence-transformers",
    "faiss-cpu",
    "numpy",
    "transformers"
]

for p in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", p])


# -----------------------------
# Imports
# -----------------------------
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import pipeline


# -----------------------------
# Step 1: Create Knowledge Sources
# -----------------------------

internal_docs = [
    "Employees are entitled to 20 days of paid leave per year.",
    "Travel reimbursement must be submitted within 7 days.",
    "The company follows a hybrid work policy."
]

faq_docs = [
    "Shipping usually takes 3 to 5 business days.",
    "Refunds are available within 30 days of purchase.",
    "Customer support is available 24 hours via email."
]


# -----------------------------
# Step 2: Load Embedding Model
# -----------------------------
model = SentenceTransformer("all-MiniLM-L6-v2")


# -----------------------------
# Step 3: Create Embeddings
# -----------------------------
internal_embeddings = model.encode(internal_docs)
faq_embeddings = model.encode(faq_docs)


# -----------------------------
# Step 4: Create Vector Databases
# -----------------------------
dim = internal_embeddings.shape[1]

internal_index = faiss.IndexFlatL2(dim)
internal_index.add(np.array(internal_embeddings))

faq_index = faiss.IndexFlatL2(dim)
faq_index.add(np.array(faq_embeddings))

print("Vector databases created")


# -----------------------------
# Step 5: Agent Router
# -----------------------------
def router(query):

    query = query.lower()

    if "employee" in query or "policy" in query or "leave" in query:
        return "internal"

    elif "shipping" in query or "refund" in query or "support" in query:
        return "faq"

    else:
        return "faq"


# -----------------------------
# Step 6: User Query
# -----------------------------
query = "What is the employee leave policy?"

print("\nUser Query:", query)


# -----------------------------
# Step 7: Agent Chooses Source
# -----------------------------
source = router(query)

print("Agent selected source:", source)


# -----------------------------
# Step 8: Retrieval
# -----------------------------
query_embedding = model.encode([query])

if source == "internal":

    distances, indices = internal_index.search(query_embedding, 2)

    context = ""
    for i in indices[0]:
        context += internal_docs[i] + "\n"

else:

    distances, indices = faq_index.search(query_embedding, 2)

    context = ""
    for i in indices[0]:
        context += faq_docs[i] + "\n"


print("\nRetrieved Context:\n")
print(context)


# -----------------------------
# Step 9: Generation
# -----------------------------
generator = pipeline("text-generation", model="gpt2")

prompt = f"""
Use the following context to answer the question.

Context:
{context}

Question: {query}

Answer:
"""

response = generator(prompt, max_length=150)

print("\nGenerated Answer:\n")
print(response[0]["generated_text"])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector databases created

User Query: What is the employee leave policy?
Agent selected source: internal

Retrieved Context:

Employees are entitled to 20 days of paid leave per year.
The company follows a hybrid work policy.



Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generated Answer:


Use the following context to answer the question.

Context:
Employees are entitled to 20 days of paid leave per year.
The company follows a hybrid work policy.


Question: What is the employee leave policy?

Answer:

Employees are entitled to 20 days of paid leave per year.

In the absence of a valid reason for the employee leave policy, the employee may choose to leave immediately after the due date of his or her return. However, all employees must be compensated for the 30 days of paid leave immediately before he or she leaves the company.

Employees may also choose to leave the company after the due date of their return if they feel that the situation is not a risk to the company.

To leave after the due date of their return:

Employees must complete an annual leave application form.

The form must be received by 6:00 a.m. on the due date of their return (or 8:00 a.m. on the morning of the due date).


Employees must present proof of their employment history.

T